In [2]:
import jax
import jax.numpy as jnp
from jax import lax, jit, grad
from jax.experimental import mesh_utils
from jax.sharding import PositionalSharding, PartitionSpec
from jax.experimental.pjit import pjit
from functools import partial
import time

# -------------------------------------------------------------------------
# Configuration & Constants
# -------------------------------------------------------------------------
MAX_RECURSION_DEPTH    = 1_000_000
OPTIMAL_DEPTH_STEP     = 250_000
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000

# Clamp bounds to reduce overflow and NaNs
VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi with scaling (with dynamic phi adaptation)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    # Adaptive scaling: for very large depths, the decay is stronger.
    adapt = jnp.maximum(1.0, depth / 100_000.0)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor * adapt + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Single 250k-step chunk with optional clamping to avoid NaNs.
# -------------------------------------------------------------------------
def single_chunk_fori_loop(x, scale_factor=1.0):
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        # Clamp value to keep it within safe range.
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val  = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 1))
        return new_val

    steps = jnp.int32(OPTIMAL_DEPTH_STEP)
    return lax.fori_loop(0, steps, body_fn, x)

# -------------------------------------------------------------------------
# 3) Chunked recursion with debug prints & dynamic phi scaling.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["total_depth", "scale_factor"])
def chunked_dppu_debug(x, total_depth, scale_factor=1.0):
    iterations = total_depth // OPTIMAL_DEPTH_STEP

    def chunk_body(chunk_idx, val):
        new_val = single_chunk_fori_loop(val, scale_factor=scale_factor)
        mean_val = jnp.mean(new_val)
        jax.debug.print(
            "Chunk {chunk_idx} => mean(new_val)={mean_val:.6f}",
            chunk_idx=chunk_idx,
            mean_val=mean_val
        )
        return new_val

    final_x = lax.fori_loop(0, iterations, chunk_body, x)
    return final_x

# -------------------------------------------------------------------------
# 4) Multi-core TPU Setup using a Device Mesh and pjit.
# -------------------------------------------------------------------------
devices = jax.devices()  # Expecting 8 TPU cores on a v2-8
mesh = jax.sharding.Mesh(devices, ('dp',))  # 'dp' for data parallel

# Create a PositionalSharding object over the mesh.
data_sharding = PositionalSharding(mesh.devices)

# Wrap the high-level processing function with pjit.
# Note: Our function has three arguments (x, total_depth, scale_factor).
# We shard 'x' across cores and pass the other scalars as-is (None).
@partial(pjit, in_shardings=(data_sharding, None, None), out_shardings=data_sharding)
def process_with_larger_depths_debug(x, total_depth, scale_factor=0.5):
    return chunked_dppu_debug(x, total_depth=total_depth, scale_factor=scale_factor)

# -------------------------------------------------------------------------
# 5) Utility functions for logging and precompilation.
# -------------------------------------------------------------------------
def log_tpu_memory(msg=""):
    print(f"[MEM DEBUG] {msg} - (Memory usage not implemented in code)")

def precompile_recursion(depths=(250_000, 500_000, 1_000_000), scale_factor=0.5):
    dummy_input = jnp.zeros((1_000,))
    for d in depths:
        print(f"Precompiling recursion for depth={d} with scale_factor={scale_factor}")
        _ = chunked_dppu_debug(dummy_input, total_depth=d, scale_factor=scale_factor).block_until_ready()

# -------------------------------------------------------------------------
# 6) Benchmark & Execution.
# -------------------------------------------------------------------------
def run_tpu_benchmarks(batch_input, depths=(250_000, 500_000, 1_000_000), scale_factor=0.5, num_trials=2):
    results = []
    for depth in depths:
        times = []
        for trial_i in range(num_trials):
            log_tpu_memory(msg=f"Before depth={depth}, trial={trial_i}")
            start_time = time.time()

            output = process_with_larger_depths_debug(batch_input, depth, scale_factor)
            out_host = jax.device_get(output)

            elapsed = time.time() - start_time
            times.append(elapsed)
            log_tpu_memory(msg=f"After depth={depth}, trial={trial_i}")
            mean_val = float(jnp.mean(out_host))
            results.append({
                "depth": depth,
                "trial": trial_i,
                "time": elapsed,
                "mean_output": mean_val
            })
        avg_time = sum(times) / len(times)
        print(f"\n🔥 TPU Benchmark (Depth={depth}, Scale={scale_factor}, Batch={BATCH_SIZE})")
        print(f"  Avg: {avg_time:.6f} sec | Min: {min(times):.6f} sec | Max: {max(times):.6f} sec")
    return results

# -------------------------------------------------------------------------
# 7) Main script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    print("Allocating batch_input...")
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)
    # Place batch_input on the TPU using data_sharding.
    batch_input = jax.device_put(batch_input, data_sharding)

    precompile_recursion(depths=[250_000, 500_000, 1_000_000], scale_factor=0.5)

    bench_results = run_tpu_benchmarks(
        batch_input,
        depths=[250_000, 500_000, 1_000_000],
        scale_factor=0.5,
        num_trials=2
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)


Allocating batch_input...
Precompiling recursion for depth=250000 with scale_factor=0.5
Precompiling recursion for depth=500000 with scale_factor=0.5
Chunk 0 => mean(new_val)=0.000000
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
Precompiling recursion for depth=1000000 with scale_factor=0.5
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
Chunk 2 => mean(new_val)=0.000000
Chunk 3 => mean(new_val)=0.000000
[MEM DEBUG] Before depth=250000, trial=0 - (Memory usage not implemented in code)


/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(


Chunk 0 => mean(new_val)=0.000000
[MEM DEBUG] After depth=250000, trial=0 - (Memory usage not implemented in code)
[MEM DEBUG] Before depth=250000, trial=1 - (Memory usage not implemented in code)
Chunk 0 => mean(new_val)=0.000000
[MEM DEBUG] After depth=250000, trial=1 - (Memory usage not implemented in code)

🔥 TPU Benchmark (Depth=250000, Scale=0.5, Batch=50000000)
  Avg: 221.878293 sec | Min: 221.710260 sec | Max: 222.046325 sec
[MEM DEBUG] Before depth=500000, trial=0 - (Memory usage not implemented in code)
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
[MEM DEBUG] After depth=500000, trial=0 - (Memory usage not implemented in code)
[MEM DEBUG] Before depth=500000, trial=1 - (Memory usage not implemented in code)
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
[MEM DEBUG] After depth=500000, trial=1 - (Memory usage not implemented in code)

🔥 TPU Benchmark (Depth=500000, Scale=0.5, Batch=50000000)
  Avg: 443.261996 sec | Min: 443.257789 se